# 🔬 CG-MedSAM Quickstart: Contrast-Gated Skin Lesion Segmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raju-sah/MedSAM-For-Skin-Segmentation/blob/main/notebooks/CG_MedSAM_Quickstart.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Model%20Card-orange)](https://huggingface.co/raju-ai/CG-MedSAM)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/raju-sah/MedSAM-For-Skin-Segmentation)
[![Project Website](https://img.shields.io/badge/Website-Showcase-emerald)](https://raju-sah.github.io/MedSAM-For-Skin-Segmentation/)

Welcome to the official quickstart notebook for **CG-MedSAM**: a contrast-conditioned parameter-efficient fine-tuning (PEFT) framework for MedSAM (Vision Transformer) designed to maintain high segmentation accuracy across diverse Fitzpatrick skin-tone groups (FST I–VI) under clean and noisy prompt conditions.

### Highlights:
- **Zero-Leakage Optical Physics:** Extracts localized CIE $L^*a^*b^*$ color distance ($\Delta E^*_{ab}$) strictly from prompt bounding boxes without accessing ground-truth masks.
- **Strict Parameter Efficiency:** Updates only **4.64%** of ViT parameters.
- **Multi-Tone Robustness:** Outperforms standard bottleneck adapters (+2.12% on dark skin, 14% disparity reduction).

## Step 1: Environment Setup & Clone Repository
We clone the official repository and install the required dependencies (PyTorch, segment-anything, and huggingface_hub).

In [ ]:
# Clone repo if executing inside Colab
import os, sys
if not os.path.exists('MedSAM-For-Skin-Segmentation') and not os.path.exists('src'):
    !git clone https://github.com/raju-sah/MedSAM-For-Skin-Segmentation.git
    %cd MedSAM-For-Skin-Segmentation

# Install core dependencies
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q huggingface_hub opencv-python matplotlib pandas

import torch
print(f"PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Download Model Checkpoints from Hugging Face Hub
We fetch the official CG-MedSAM adapter weights directly from the [Hugging Face Model Hub](https://huggingface.co/raju-ai/CG-MedSAM).

In [ ]:
from huggingface_hub import hf_hub_download
import os

checkpoints_dir = 'checkpoints'
os.makedirs(checkpoints_dir, exist_ok=True)

print("Fetching CG-MedSAM weights from Hugging Face...")
adapter_path = hf_hub_download(
    repo_id="raju-ai/CG-MedSAM",
    filename="best_cg_adapter_model.pth",
    local_dir=checkpoints_dir
)
print(f"[✓] CG-Adapter checkpoint ready: {adapter_path}")

# Also download MedSAM ViT-B base checkpoint if needed
base_medsam_path = os.path.join(checkpoints_dir, "medsam_vit_b.pth")
if not os.path.exists(base_medsam_path):
    print("Downloading foundation MedSAM ViT-B checkpoint from raju-ai/CG-MedSAM...")
    hf_hub_download(
        repo_id="raju-ai/CG-MedSAM",
        filename="medsam_vit_b.pth",
        local_dir=checkpoints_dir
    )

print("Checkpoints verified:", os.listdir(checkpoints_dir))

## Step 3: Initialize CG-MedSAM Engine
We load the unified PEFT MedSAM model, activate the bottleneck adapters, and bind the Contrast-Gating MLP.

In [ ]:
import sys
sys.path.insert(0, '.')

from src.eval.inference_utils import load_model, run_inference, compute_lab_contrast_proxy, auto_detect_prompt_bbox

# Hardware detection with compute capability verification
device = torch.device('cpu')
if torch.cuda.is_available():
    try:
        _test = torch.zeros(1, device='cuda')
        _ = _test + 1
        device = torch.device('cuda')
        print(f"[✓] Compute device: {torch.cuda.get_device_name(0)} (CUDA Enabled)")
    except Exception as e:
        print(f"[!] Notice: CUDA device not compatible with runtime ({e}). Falling back to CPU.")
        device = torch.device('cpu')
else:
    print("[i] Compute device: CPU")

model, is_mock = load_model(model_name='cg_adapter', checkpoint_path=adapter_path, base_checkpoint_path=base_medsam_path, device=device)
print(f"[✓] CG-MedSAM model successfully initialized (is_mock={is_mock})")

## Step 4: Run Inference Across Fitzpatrick Skin Tone Samples
We test the model across three pre-packaged clinical samples spanning:
1. **Light Skin (FST I–II):** High lesion-to-skin color contrast.
2. **Medium Skin (FST III–IV):** Moderate contrast.
3. **Dark Skin (FST V–VI):** Low contrast, challenging lesion boundary.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

sample_paths = [
    ("demo_samples/sample_light_fst_ii.jpg", "Light Cohort (FST I–II)"),
    ("demo_samples/sample_medium_fst_iv.jpg", "Medium Cohort (FST III–IV)"),
    ("demo_samples/sample_dark_fst_v.jpg", "Dark Cohort (FST V–VI)")
]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
plt.subplots_adjust(wspace=0.15, hspace=0.25)

for row_idx, (path, label) in enumerate(sample_paths):
    bgr = cv2.imread(path)
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    
    # 1. Automatically propose clinician bounding box
    bbox = auto_detect_prompt_bbox(rgb)
    
    # 2. Run CG-MedSAM forward pass with zero-leakage proxy calculation
    result = run_inference(model, rgb, bbox, device=device)
    mask = result["mask"]
    c_info = result["contrast_info"]
    
    # Panel (a): Original image with prompt box
    ax0 = axes[row_idx, 0]
    ax0.imshow(rgb)
    x1, y1, x2, y2 = bbox
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='gold', linewidth=2.5, linestyle='--')
    ax0.add_patch(rect)
    ax0.set_title(f"{label}\nPrompt Box (Otsu)", fontsize=11, fontweight='bold')
    ax0.axis('off')
    
    # Panel (b): Optical Physics Telemetry Gauge
    ax1 = axes[row_idx, 1]
    ax1.set_facecolor('#0B0F19')
    ax1.axis('off')
    delta_e = c_info['delta_e']
    gamma = c_info['gamma_factor']
    fst = c_info['estimated_fst']
    ita = c_info['ita_degrees']
    ax1.text(0.1, 0.75, r"$\Delta E^*_{{ab}}$ Contrast: " + f"{delta_e:.2f}", color='#00F0FF', fontsize=12, fontweight='bold')
    ax1.text(0.1, 0.55, r"Gate Factor $\gamma$: " + f"{gamma:.3f}", color='#A855F7', fontsize=12, fontweight='bold')
    ax1.text(0.1, 0.35, f"Estimated FST: {fst}", color='#FBBF24', fontsize=11)
    ax1.text(0.1, 0.15, f"ITA Angle: {ita:.1f}°", color='#CBD5E1', fontsize=11)
    ax1.set_title("Optical Physics Gauge", fontsize=11, fontweight='bold')
    
    # Panel (c): Predicted Binary Mask
    ax2 = axes[row_idx, 2]
    ax2.imshow(mask, cmap='gray')
    ax2.set_title("CG-MedSAM Mask", fontsize=11, fontweight='bold')
    ax2.axis('off')
    
    # Panel (d): High-Contrast Boundary Overlay
    ax3 = axes[row_idx, 3]
    overlay = rgb.copy()
    overlay[mask == 1] = (0.5 * overlay[mask == 1] + 0.5 * np.array([239, 68, 68])).astype(np.uint8)
    ax3.imshow(overlay)
    ax3.set_title("Segmentation Overlay", fontsize=11, fontweight='bold')
    ax3.axis('off')

plt.tight_layout()
plt.show()

## Step 5: Test on Your Own Image
Upload a clinical photo or dermoscopic image and provide custom bounding-box coordinates to test CG-MedSAM on arbitrary clinical data.

In [ ]:
from google.colab import files
import io

print("Upload your custom image (or skip to test with default):")
try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        img_bytes = uploaded[filename]
        nparr = np.frombuffer(img_bytes, np.uint8)
        user_rgb = cv2.cvtColor(cv2.imdecode(nparr, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        
        user_bbox = auto_detect_prompt_bbox(user_rgb)
        res = run_inference(model, user_rgb, user_bbox, device=device)
        
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(user_rgb)
        x1, y1, x2, y2 = user_bbox
        plt.gca().add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='cyan', linewidth=2))
        plt.title("Input + Auto Prompt")
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.imshow(res['mask'], cmap='jet')
        plt.title(f"CG-MedSAM Mask (ΔE*={res['contrast_info']['delta_e']:.1f}, γ={res['contrast_info']['gamma_factor']:.2f})")
        plt.axis('off')
        plt.show()
except Exception as e:
    print("Interactive upload skipped or running in non-Colab environment.", e)

## Citation & Links
If you use CG-MedSAM in your research, please cite our MICCAI 2026 work:
```bibtex
@inproceedings{sah2026cgmedsam,
  title={Contrast-Gated Parameter-Efficient Adaptation of MedSAM for Skin-Tone-Robust Lesion Segmentation},
  author={Sah, Raju and Consortium, Anonymous Medical AI Research},
  booktitle={International Conference on Medical Image Computing and Computer-Assisted Intervention (MICCAI)},
  year={2026},
  organization={Springer}
}
```
- **GitHub Repository:** [https://github.com/raju-sah/MedSAM-For-Skin-Segmentation](https://github.com/raju-sah/MedSAM-For-Skin-Segmentation)
- **Model Hub:** [https://huggingface.co/raju-ai/CG-MedSAM](https://huggingface.co/raju-ai/CG-MedSAM)
- **Research Showcase:** [https://raju-sah.github.io/MedSAM-For-Skin-Segmentation/](https://raju-sah.github.io/MedSAM-For-Skin-Segmentation/)
- **Live Interactive Demo:** [https://huggingface.co/spaces/raju-ai/CG-MedSAM-Showcase](https://huggingface.co/spaces/raju-ai/CG-MedSAM-Showcase)